# MECG: Multilingual Evidence-Grounded Complaint Generation
## Notebook 06: Aspect & Emotion Classifier Training

**Goal**: Train supervised text classifiers using `microsoft/deberta-v3-base` on the leak-free splits of `ComVID_cleaned.csv`:
1. **Aspect Multi-Label Classifier**: Identifies multiple complaint aspects (Quality, Usability, Manufacturing, Functionality, etc.) using `BCEWithLogitsLoss` with positive class weight scaling.
2. **Emotion Multi-Class Classifier**: Predicts customer emotion across the 4 canonical classes (`blame`, `disappointment`, `dissatisfaction`, `frustration`) using weighted `CrossEntropyLoss`.
3. **Leakage Isolation**: Strictly uses `split == 'train'` for training and `split == 'val'` for validation.
4. **Evaluation**: Evaluates Micro-F1, Macro-F1, and per-class metrics.

In [ ]:
# Environment & Dependencies Verification
import os
import sys
import json
from pathlib import Path

CURRENT_DIR = Path.cwd()
if str(CURRENT_DIR) not in sys.path:
    sys.path.append(str(CURRENT_DIR))
if str(CURRENT_DIR.parent) not in sys.path:
    sys.path.append(str(CURRENT_DIR.parent))

import torch
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name     : {torch.cuda.get_device_name(0)}')

from src.classification.aspect_classifier import AspectClassifier
from src.classification.emotion_classifier import EmotionClassifier
print('Classification modules loaded successfully!')

### 1. Inspect Training and Validation Data Splits
We verify that `split == 'train'` (940 samples) and `split == 'val'` (118 samples) are loaded without any video-level data leakage.

In [ ]:
import csv

CLEANED_CSV = Path('data/processed/ComVID_cleaned.csv')
if not CLEANED_CSV.exists() and Path('/kaggle/input/comvid-dataset/ComVID_cleaned.csv').exists():
    CLEANED_CSV = Path('/kaggle/input/comvid-dataset/ComVID_cleaned.csv')

train_samples = 0
val_samples = 0
with open(CLEANED_CSV, 'r', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        if row['split'] == 'train':
            train_samples += 1
        elif row['split'] == 'val':
            val_samples += 1

print(f'Dataset path : {CLEANED_CSV.resolve()}')
print(f'Train split  : {train_samples} samples')
print(f'Val split    : {val_samples} samples')

### 2. Train DeBERTa-v3 Multi-Label Aspect Classifier
Executes supervised training using `training/train_aspect_classifier.py`.

In [ ]:
from training.train_aspect_classifier import train as train_aspect

aspect_checkpoint_dir = 'outputs/checkpoints/aspect_classifier'
print('Training Aspect Classifier (5 epochs)...')
train_aspect(
    csv_path=str(CLEANED_CSV),
    output_dir=aspect_checkpoint_dir,
    model_id='microsoft/deberta-v3-base',
    epochs=5,
    batch_size=16,
    lr=2e-5,
    dry_run=False
)
print('Aspect Classifier Training Completed!')

### 3. Train DeBERTa-v3 Emotion Classifier
Executes supervised training using `training/train_emotion_classifier.py`.

In [ ]:
from training.train_emotion_classifier import train as train_emotion

emotion_checkpoint_dir = 'outputs/checkpoints/emotion_classifier'
print('Training Emotion Classifier (5 epochs)...')
train_emotion(
    csv_path=str(CLEANED_CSV),
    output_dir=emotion_checkpoint_dir,
    model_id='microsoft/deberta-v3-base',
    epochs=5,
    batch_size=16,
    lr=2e-5,
    dry_run=False
)
print('Emotion Classifier Training Completed!')

### 4. Test Classifier Predictions
Test both trained models on sample customer complaint texts.

In [ ]:
sample_review = 'The plastic bottle cap is completely loose and leaks pungent chemical liquid everywhere.'

aspect_clf = AspectClassifier.from_checkpoint(aspect_checkpoint_dir)
aspect_preds = aspect_clf.predict(sample_review, threshold=0.4)
print(f"Review: '{sample_review}'")
print(f"Predicted Aspects: {aspect_preds}")

emotion_clf = EmotionClassifier(checkpoint_dir=emotion_checkpoint_dir)
emotion_preds = emotion_clf.predict(sample_review)
print(f"Predicted Emotion: {emotion_preds['predicted_emotion']} (confidence={emotion_preds['confidence']})")
print(f"Emotion Distribution: {emotion_preds['distribution']}")

### Summary & Next Steps
- Aspect and Emotion classifiers are trained, evaluated, and checkpointed.
- Proceed to **Phase 6: Visual Evidence Extraction** and **Phase 7: Visual Retrieval with SigLIP 2 & FAISS**.